# S09: Variant consistency check (no-FTS variant vs main v1.0.1)

Report only. Reads the variant feature log (`config.tables.feature_log`) and the main v1.0.1 feature log (read-only),
and writes only under the variant `artifacts_root` (`reports/99_variant_consistency.xlsx`, `reports/run_summaries/s09_variant_consistency.json`).

1. Every FTS feature has `L0_reason = SOURCE_EXCLUDED_BY_SCOPE` in the variant (asserted).
2. For every other feature the L0–L3 columns are identical in both logs, tolerance 1e-9 (asserted).
3. Features whose L4 / L5 outcome differs between the runs, with the main run's `corr_with` (report only).
4. `funnel_config_hash` equals the v1.0.1 value from the locked main registry (asserted).

In [0]:
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" openpyxl --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, time, hashlib, shutil, tempfile, importlib
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)

ARTIFACTS      = common.artifacts_path(cfg)          # variant root: the only place this notebook writes
MAIN_ARTIFACTS = common.main_artifacts_path(cfg)     # main v1.0.1 root: read-only
REPORTS        = os.path.join(ARTIFACTS, "reports")
RUN_SUMS       = os.path.join(REPORTS, "run_summaries")
XLSX_PATH      = os.path.join(REPORTS, "09_variant_consistency.xlsx")
SUMMARY_PATH   = os.path.join(RUN_SUMS, "s09_variant_consistency.json")

IS_VARIANT = common.variant_guard(cfg)               # artifacts_root / feature log differ from the main model
assert IS_VARIANT, "s09 compares a variant with the main model, but config is not a variant run " \
                   "(artifacts_root == main_artifacts_root)"
for p in (XLSX_PATH, SUMMARY_PATH):
    assert p.startswith(ARTIFACTS.rstrip("/") + "/") and not p.startswith(MAIN_ARTIFACTS.rstrip("/") + "/"), \
        f"output {p} is not under the variant artifacts_root"
os.makedirs(RUN_SUMS, exist_ok=True)
RUN_CTX = common.print_run_context(cfg, PROJECT_ROOT)

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
VAR_FLOG_TBL  = common.table_name(cfg, "feature_log")
MAIN_FLOG_REQ = f"{cat}.{sch}.acard_feature_log_v101"    # main log named in the request
MAIN_FLOG_DEF = common.main_table_name(cfg, "feature_log")  # main log written by v1.0.1 (default name)
MAIN_REGISTRY_PATH = os.path.join(MAIN_ARTIFACTS, "model_registry.json")

TOL = 1e-9
STRICT_COLS = ["L0_pass", "L0_reason", "single_value_share", "missing_rate", "iv_dev", "iv_cv", "n_bins", "trend",
               "direction_conflict", "iv_s1", "iv_s2", "iv_s3", "iv_s4", "rank_corr_min", "head_tail_flip", "L2_flag",
               "L2_pass", "psi_max_dev", "psi_last_q", "max_adj_psi", "psi_fwd", "psi_status", "cause_type",
               "psi_decision", "L3_pass"]
L45_OUTCOME = ["L4_pass", "L5_pass"]                   # outcome columns compared for point 3
L45_DETAIL  = ["L4_reason", "L5_reason"]

GIT_COMMIT  = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
FUNNEL_HASH = hashlib.sha256(json.dumps(cfg["funnel"], sort_keys=True).encode()).hexdigest()
_TMP = tempfile.mkdtemp(prefix="s09_")
_t0 = time.time()
print(f"Git commit : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]} | Funnel hash: {FUNNEL_HASH[:12]}")
print(f"Variant log: {VAR_FLOG_TBL}")
print(f"Report     : {XLSX_PATH}")

In [0]:
# ── Main registry (locked v1.0.1) is the reference for the main log and the funnel hash ──
with open(MAIN_REGISTRY_PATH) as fh:
    MAIN_REG = json.load(fh)
assert MAIN_REG.get("locked") is True, f"main registry {MAIN_REGISTRY_PATH} is not locked"
print(f"Main registry: v{MAIN_REG['version']} locked at {MAIN_REG.get('locked_at')}")
vc_main = (cfg.get("variant_comparison") or {}).get("main_registry")
if vc_main and os.path.abspath(vc_main) != os.path.abspath(MAIN_REGISTRY_PATH):
    print(f"\u26a0 config.variant_comparison.main_registry = {vc_main} differs from {MAIN_REGISTRY_PATH} "
          f"(exists: {os.path.exists(vc_main)}); this check uses <main_artifacts_root>/model_registry.json")

# ── Main feature log (read-only). acard_feature_log_v101 if it exists, else the default main log, which is
#    accepted only if it reproduces the locked registry (Model A features and B/C pool). ──
MAIN_FLOG_TBL = MAIN_FLOG_REQ if spark.catalog.tableExists(MAIN_FLOG_REQ) else MAIN_FLOG_DEF
if MAIN_FLOG_TBL != MAIN_FLOG_REQ:
    print(f"\u26a0 {MAIN_FLOG_REQ} does not exist; using {MAIN_FLOG_TBL} (the log written by the v1.0.1 run)")
main_log = spark.table(MAIN_FLOG_TBL).toPandas()
var_log  = spark.table(VAR_FLOG_TBL).toPandas()
for df in (main_log, var_log):
    df.attrs = {}
    assert df["feature"].is_unique, "feature column is not unique in a feature log"

main_A    = set(main_log.loc[main_log["final_A"].astype(str).str.upper() == "Y", "feature"]) if "final_A" in main_log else set()
main_pool = set(main_log.loc[main_log["L4_pass"].astype(str).str.upper() == "Y", "feature"]) if "L4_pass" in main_log else set()
REG_A, REG_POOL = set(MAIN_REG["features"]["A"]), set(MAIN_REG["features"]["BC_pool"])
MAIN_LOG_MATCHES_REGISTRY = (main_A == REG_A) and (main_pool == REG_POOL)
print(f"Main log {MAIN_FLOG_TBL}: {len(main_log)} rows | final_A {len(main_A)} vs registry A {len(REG_A)} | "
      f"L4 pool {len(main_pool)} vs registry {len(REG_POOL)} -> matches registry: {MAIN_LOG_MATCHES_REGISTRY}")
print(f"Variant log {VAR_FLOG_TBL}: {len(var_log)} rows, {len(var_log.columns)} columns")
SAME_FEATURES = set(main_log["feature"]) == set(var_log["feature"])
print(f"Same feature rows in both logs: {SAME_FEATURES} "
      f"(only main {len(set(main_log.feature) - set(var_log.feature))}, only variant {len(set(var_log.feature) - set(main_log.feature))})")

M = main_log.set_index("feature")
V = var_log.set_index("feature")
COMMON = sorted(set(M.index) & set(V.index))

In [0]:
# FTS features: config.feature_scope.exclude_prefixes applied to feature_dict (prefix / source / description, same rule
# as E4) plus any feature-log row whose name contains the prefix stem or whose source is the stem (e.g. FTS).
fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
scope = common.scope_exclusions(fdict[fdict["role"] == "FEATURE"], cfg)
prefixes = [str(p).lower() for p in ((cfg.get("feature_scope") or {}).get("exclude_prefixes") or []) if p]
assert prefixes, "config.feature_scope.exclude_prefixes is empty: no FTS scope defined for the variant"
stems = sorted({p.strip("_") for p in prefixes if p.strip("_")})

by_log = set(V.index[[any(s in f.lower() for s in stems) for f in V.index]]) | \
         set(V.index[V["source"].astype(str).str.strip().str.lower().isin(stems)])
FTS = sorted(set(scope["feature"]) | by_log)
basis = scope.set_index("feature")["basis"].to_dict()

fts_rows = pd.DataFrame({
    "feature": FTS,
    "basis": [basis.get(f, "feature_log name/source") for f in FTS],
    "in_variant_log": [f in V.index for f in FTS],
    "variant_L0_pass": [V.at[f, "L0_pass"] if f in V.index else None for f in FTS],
    "variant_L0_reason": [V.at[f, "L0_reason"] if f in V.index else None for f in FTS],
    "main_L0_reason": [M.at[f, "L0_reason"] if f in M.index else None for f in FTS],
})
fts_rows["ok"] = fts_rows["in_variant_log"] & (fts_rows["variant_L0_reason"] == common.SCOPE_EXCLUDED_REASON) \
                 & (fts_rows["variant_L0_pass"] == "N")
CHECK1_OK = bool(len(fts_rows) > 0 and fts_rows["ok"].all())
print(f"FTS features: {len(FTS)} (by basis: {fts_rows['basis'].value_counts().to_dict()})")
print(f"Check 1 — all have L0_reason = {common.SCOPE_EXCLUDED_REASON}: {CHECK1_OK} "
      f"({int((~fts_rows['ok']).sum())} not)")
if not CHECK1_OK:
    display(fts_rows[~fts_rows["ok"]])

In [0]:
_NULLS = {"", "nan", "none", "nat", "<na>", "null"}

def _norm(v):
    """Null-like values (None, NaN, 'nan', 'None', '') -> None; other values -> stripped string."""
    if v is None or (isinstance(v, float) and np.isnan(v)) or v is pd.NA:
        return None
    s = str(v).strip()
    return None if s.lower() in _NULLS else s

def _as_num(s):
    """Numeric version of a column if every non-null value is numeric, else None."""
    n = s.map(_norm)
    out = pd.to_numeric(n, errors="coerce")
    return out if (n.isna() | out.notna()).all() else None

OTHER = [f for f in COMMON if f not in set(FTS)]
miss_cols = [c for c in STRICT_COLS if c not in M.columns or c not in V.columns]
mismatch_rows, col_summary = [], []
for c in [c for c in STRICT_COLS if c not in miss_cols]:
    m, v = M.loc[OTHER, c], V.loc[OTHER, c]
    mn, vn = _as_num(m), _as_num(v)
    numeric = (mn is not None and vn is not None
               and (pd.api.types.is_numeric_dtype(m) or pd.api.types.is_numeric_dtype(v)))
    if numeric:
        diff = (mn - vn).abs()
        eq = (mn.isna() & vn.isna()) | (diff <= TOL)
    else:
        diff = pd.Series(np.nan, index=m.index)
        eq = m.map(_norm) == v.map(_norm)
        eq |= m.map(_norm).isna() & v.map(_norm).isna()
    col_summary.append({"column": c, "type": "numeric" if numeric else "text", "n_compared": len(OTHER),
                        "n_mismatch": int((~eq).sum()),
                        "max_abs_diff": float(diff.max()) if numeric and diff.notna().any() else None})
    for f in eq.index[~eq]:
        mismatch_rows.append({"feature": f, "column": c, "main": m.at[f], "variant": v.at[f],
                              "abs_diff": float(diff.at[f]) if numeric and pd.notna(diff.at[f]) else None})

col_summary = pd.DataFrame(col_summary)
mismatches = pd.DataFrame(mismatch_rows, columns=["feature", "column", "main", "variant", "abs_diff"])
CHECK2_OK = bool(SAME_FEATURES and not miss_cols and len(mismatches) == 0)
print(f"Non-FTS features compared: {len(OTHER)} on {len(STRICT_COLS) - len(miss_cols)} columns (tolerance {TOL})")
if miss_cols:
    print(f"\u274c Columns missing from a log: {miss_cols}")
print(f"Check 2 — identical: {CHECK2_OK} ({len(mismatches)} mismatching cells, "
      f"{mismatches['feature'].nunique() if len(mismatches) else 0} features)")
display(col_summary)
if len(mismatches):
    display(mismatches.head(50))

In [0]:
VAR_L45_RUN = all(c in V.columns for c in L45_OUTCOME)
L45_COLS = ["feature", "is_fts"] + [f"{s}_{c}" for c in L45_OUTCOME + L45_DETAIL for s in ("main", "variant")] \
           + ["main_corr_with", "main_corr_r"]
if VAR_L45_RUN:
    rows = []
    for f in COMMON:
        mo = [_norm(M.at[f, c]) if c in M.columns else None for c in L45_OUTCOME]
        vo = [_norm(V.at[f, c]) for c in L45_OUTCOME]
        if mo != vo:
            r = {"feature": f, "is_fts": f in set(FTS)}
            for c in L45_OUTCOME + L45_DETAIL:
                r[f"main_{c}"] = _norm(M.at[f, c]) if c in M.columns else None
                r[f"variant_{c}"] = _norm(V.at[f, c]) if c in V.columns else None
            r["main_corr_with"] = _norm(M.at[f, "corr_with"]) if "corr_with" in M.columns else None
            r["main_corr_r"] = M.at[f, "corr_r"] if "corr_r" in M.columns else None
            rows.append(r)
    l45_diff = pd.DataFrame(rows, columns=L45_COLS)
    L45_NOTE = f"{len(l45_diff)} feature(s) with a different L4/L5 outcome ({int(l45_diff['is_fts'].sum()) if len(l45_diff) else 0} FTS)"
else:
    l45_diff = pd.DataFrame(columns=L45_COLS)
    L45_NOTE = (f"Variant L4/L5 not run yet: {[c for c in L45_OUTCOME if c not in V.columns]} missing from "
                f"{VAR_FLOG_TBL}. Re-run s09 after s03_L4 and s03_L5 on the variant.")
print(f"Check 3 (report only) — {L45_NOTE}")
if len(l45_diff):
    display(l45_diff)

In [0]:
V101_FUNNEL_HASH = MAIN_REG["funnel_config_hash"]
hash_rows = [{"source": "main registry v" + str(MAIN_REG["version"]), "funnel_config_hash": V101_FUNNEL_HASH},
             {"source": "current config.funnel (variant)", "funnel_config_hash": FUNNEL_HASH}]
for fn in sorted(os.listdir(RUN_SUMS)):
    if fn.startswith("s03_") and fn.endswith(".json"):
        with open(os.path.join(RUN_SUMS, fn)) as fh:
            h = json.load(fh).get("funnel_config_hash")
        if h:
            hash_rows.append({"source": f"variant run summary {fn}", "funnel_config_hash": h})
hash_df = pd.DataFrame(hash_rows)
hash_df["equals_v101"] = hash_df["funnel_config_hash"] == V101_FUNNEL_HASH
CHECK4_OK = bool(hash_df["equals_v101"].all() and len(hash_df) >= 3)   # registry + config + at least E4 summary
print(f"Check 4 — funnel_config_hash equals v1.0.1 ({V101_FUNNEL_HASH[:12]}): {CHECK4_OK}")
display(hash_df)

In [0]:
summary_df = pd.DataFrame([
    {"check": "1. FTS features have L0_reason = SOURCE_EXCLUDED_BY_SCOPE", "asserted": True, "pass": CHECK1_OK,
     "detail": f"{len(FTS)} FTS features, {int((~fts_rows['ok']).sum())} not excluded"},
    {"check": "2. L0-L3 columns identical for non-FTS features (tol 1e-9)", "asserted": True, "pass": CHECK2_OK,
     "detail": f"{len(OTHER)} features x {len(STRICT_COLS) - len(miss_cols)} columns, {len(mismatches)} mismatches"
               + (f", missing columns {miss_cols}" if miss_cols else "") + ("" if SAME_FEATURES else ", feature rows differ")},
    {"check": "3. L4/L5 outcome differences (with main corr_with)", "asserted": False, "pass": None, "detail": L45_NOTE},
    {"check": "4. funnel_config_hash equals v1.0.1", "asserted": True, "pass": CHECK4_OK,
     "detail": f"v1.0.1 {V101_FUNNEL_HASH[:12]}; {int((~hash_df['equals_v101']).sum())} source(s) differ"},
])
meta = pd.DataFrame([{
    "module": "S09_variant_consistency", "run_time": datetime.now().isoformat(timespec="seconds"),
    "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH, "branch": RUN_CTX["branch"],
    "variant_artifacts_root": ARTIFACTS, "main_artifacts_root": MAIN_ARTIFACTS,
    "variant_feature_log": VAR_FLOG_TBL, "main_feature_log": MAIN_FLOG_TBL,
    "main_log_matches_registry": MAIN_LOG_MATCHES_REGISTRY, "main_registry_version": MAIN_REG["version"],
    "variant_registry_version": RUN_CTX["registry_version"], "tolerance": TOL,
}])

tmp_xl = os.path.join(_TMP, "99_variant_consistency.xlsx")
with pd.ExcelWriter(tmp_xl, engine="openpyxl") as w:
    meta.to_excel(w, sheet_name="meta", index=False)
    summary_df.to_excel(w, sheet_name="summary", index=False)
    fts_rows.to_excel(w, sheet_name="1_fts_exclusion", index=False)
    col_summary.to_excel(w, sheet_name="2_column_summary", index=False)
    mismatches.to_excel(w, sheet_name="2_mismatches", index=False)
    (l45_diff if len(l45_diff) else pd.DataFrame({"note": [L45_NOTE]})).to_excel(w, sheet_name="3_l4_l5_differences", index=False)
    hash_df.to_excel(w, sheet_name="4_funnel_hash", index=False)
shutil.copyfile(tmp_xl, XLSX_PATH)
print(f"Saved report: {XLSX_PATH}")
display(summary_df)

In [0]:
# ── Acceptance (asserts 1, 2 and 4; point 3 is report only) ──
run_summary = {
    "module": "S09_variant_consistency", "run_time": datetime.now().isoformat(timespec="seconds"),
    "elapsed_s": round(time.time() - _t0, 1), "git_commit": GIT_COMMIT, "config_hash": CONFIG_HASH,
    "funnel_config_hash": FUNNEL_HASH, "v101_funnel_config_hash": V101_FUNNEL_HASH,
    "main_feature_log": MAIN_FLOG_TBL, "variant_feature_log": VAR_FLOG_TBL,
    "main_log_matches_registry": MAIN_LOG_MATCHES_REGISTRY,
    "n_features": len(COMMON), "n_fts": len(FTS), "n_other": len(OTHER),
    "check1_fts_scope_excluded": CHECK1_OK, "check2_l0_l3_identical": CHECK2_OK,
    "check2_mismatches": len(mismatches), "check3_note": L45_NOTE, "check3_n_differences": len(l45_diff),
    "check4_funnel_hash": CHECK4_OK, "report": XLSX_PATH,
    "pass_fail": "PASS" if (CHECK1_OK and CHECK2_OK and CHECK4_OK and MAIN_LOG_MATCHES_REGISTRY) else "FAIL",
    "run_context": RUN_CTX,
}
_tmp_js = os.path.join(_TMP, "s09.json")
with open(_tmp_js, "w") as fh:
    json.dump(run_summary, fh, indent=2, default=str)
shutil.copyfile(_tmp_js, SUMMARY_PATH)
print(f"Run summary: {SUMMARY_PATH}")

assert MAIN_LOG_MATCHES_REGISTRY, (f"{MAIN_FLOG_TBL} does not reproduce the locked v{MAIN_REG['version']} registry "
                                   f"(Model A / B-C pool); it cannot be used as the main log")
assert CHECK1_OK, f"Check 1 failed: {int((~fts_rows['ok']).sum())} FTS feature(s) without L0_reason = {common.SCOPE_EXCLUDED_REASON}"
assert CHECK2_OK, (f"Check 2 failed: {len(mismatches)} mismatching cell(s) in L0-L3 columns for non-FTS features"
                   + (f"; missing columns {miss_cols}" if miss_cols else "") + ("" if SAME_FEATURES else "; feature rows differ"))
assert CHECK4_OK, f"Check 4 failed: funnel_config_hash differs from v1.0.1 ({V101_FUNNEL_HASH[:12]})"
print("\u2705 S09 checks 1, 2 and 4 passed.", L45_NOTE)

## Reject analysis (spec Chapter 12, features-only branch): models A and C

Report only. Reads the locked 1.0.1-nofts registry, its model, binning and calibration files, `config.tables.master` (read-only) and `config.reject_inference.table`. Every output goes under `<artifacts_root>/<reject_inference.output_folder>/` (`ri_reject_analysis.xlsx`, `open_questions.md`, `ri_run_summary.json`).

* Declined = `final_decision = DECLINE` and the reason text contains INTERNAL SCORE DECLINE (selected here; the table is not pre-filtered).
* Application-time filters: verified where the column exists; the others are reported as NOT_VERIFIABLE (owner decision 2026-11-11).
* Unmapped decline reasons: report-only deviation. They are treated as PENDING and Steps 2-3 run under S1 (PENDING does not exclude) and S2 (PENDING excludes). Reasons with Dectech outcome A (approve) are NEUTRAL.
* Models: A (registry selection) and C, side by side, scored with the E13 scoring code, Platt calibration and PDO scaling.

In [0]:
# ── RI-1 Setup: write guard, registry lock and file hashes, locked models A and C ──
import pickle, copy, warnings
import lightgbm as lgb
from pyspark.sql import functions as F
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="Missing values detected")

RI_CFG = cfg["reject_inference"]
RI_DIR = os.path.abspath(os.path.join(ARTIFACTS, RI_CFG["output_folder"]))
RI_XLSX = os.path.join(RI_DIR, "ri_reject_analysis.xlsx")
RI_SUMMARY = os.path.join(RI_DIR, "ri_run_summary.json")
RI_OQ = os.path.join(RI_DIR, "open_questions.md")
os.makedirs(RI_DIR, exist_ok=True)
RI_WRITTEN = []
RI_TMP = tempfile.mkdtemp(prefix="s09_ri_")

def ri_out(path):
    """Every RI output path goes through here: it must sit under <artifacts_root>/<reject_inference.output_folder>/."""
    ap = os.path.abspath(path)
    assert ap.startswith(RI_DIR + os.sep), f"RI output {ap} is outside {RI_DIR}"
    RI_WRITTEN.append(ap)
    return ap

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(p):
    with open(p) as fh:
        return json.load(fh)

def tree_snapshot(root, skip):
    """(size, mtime) of every file under root, except the RI output folder."""
    snap = {}
    for dp, _, fns in os.walk(root):
        adp = os.path.abspath(dp)
        if adp == skip or adp.startswith(skip + os.sep):
            continue
        for fn in fns:
            p = os.path.join(adp, fn)
            st = os.stat(p)
            snap[p] = (st.st_size, st.st_mtime)
    return snap

# Registry: locked, configured version, sign-off filled, every locked file and the E13 calibration file unchanged
REGISTRY_PATH = os.path.join(ARTIFACTS, "model_registry.json")
REG_SHA0 = sha256_file(REGISTRY_PATH)
REG = read_json(REGISTRY_PATH)
assert REG.get("locked") is True, f"ABORT: {REGISTRY_PATH} is not locked"
assert REG["version"] == cfg["registry"]["version"], f"ABORT: registry {REG['version']} != config {cfg['registry']['version']}"
for k in ("decision_rules_signed_by", "signed_date"):
    assert str(REG.get("signoff", {}).get(k, "")).strip(), f"ABORT: registry sign-off field {k} is empty"
FILES = {k: v["path"] for k, v in REG["files"].items()}
HASH0 = {k: sha256_file(p) for k, p in FILES.items()}
bad = [k for k, v in REG["files"].items() if HASH0[k] != v["sha256"]]
assert not bad, f"ABORT: {len(bad)} locked file(s) changed since the lock: {bad}"
OOT_EVAL = REG.get("oot_evaluation") or {}
assert OOT_EVAL.get("selected_model"), "ABORT: registry has no oot_evaluation.selected_model (run E13 first)"
CAL_FILE = OOT_EVAL["calibration_file"]
CAL_SHA0 = sha256_file(CAL_FILE["path"])
assert CAL_SHA0 == CAL_FILE["sha256"], f"ABORT: {CAL_FILE['path']} changed since E13"
SELECTED, SEL_STATUS = OOT_EVAL["selected_model"], OOT_EVAL.get("status")
RI_MODELS = ["A", "C"]          # owner request 2026-11-11: registry selection (A) and C, side by side
SNAP0 = tree_snapshot(ARTIFACTS, RI_DIR)

# Locked models, loaded exactly as in E13 (s05_oot)
RI_FDICT = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
SPEC = read_json(FILES["models/model_spec.json"])
WOE_KW = SPEC["A"]["woe_transform"]
A_FEATS, POOL, BC_COLS = REG["features"]["A"], REG["features"]["BC_pool"], REG["features"]["BC_columns"]
with open(FILES["models/A/binning_process.pkl"], "rb") as fh:
    A_BP = pickle.load(fh)
with open(FILES["models/A/lr_sklearn.pkl"], "rb") as fh:
    A_LR = pickle.load(fh)
C_BST = lgb.Booster(model_file=FILES["models/C/lgbm.txt"])
assert C_BST.num_trees() == REG["lightgbm_rounds"]["final_rounds"], "LightGBM trees != locked final rounds"
PLATT = read_json(CAL_FILE["path"])["params"]
L1 = {}
for f in POOL:
    with open(FILES[f"binning/L1/{f}.pkl"], "rb") as fh:
        L1[f] = pickle.load(fh)
MODEL_FEATS = {"A": list(A_FEATS), "C": list(POOL)}
print(f"Registry v{REG['version']} locked {REG['locked_at']}; {len(HASH0)} file hashes + calibration file verified")
print(f"Selected model (E13): {SELECTED} [{SEL_STATUS}] | analysed: {RI_MODELS} | features A {len(A_FEATS)}, C {len(POOL)}")
print(f"RI output folder: {RI_DIR} | files outside it snapshotted: {len(SNAP0)}")

In [0]:
# ── RI Step 1.1-1.2: declined intake, E2b text normalisation, verification waterfall ──
id_col, date_col, status_col = cfg["columns"]["id"], cfg["columns"]["date"], cfg["columns"]["status"]
cust_col, dec_time_col = cfg["columns"]["customer"], cfg["columns"]["decision_time"]
LABELLED, PSI_STATUSES, EPS = cfg["labelled_statuses"], cfg["psi_statuses"], cfg["funnel"]["L3"]["eps"]
REASON_COL, DELIM = RI_CFG["reason_column"], RI_CFG["reason_delimiter"]
D_ID, D_DATE, D_CUST, D_DOB = "application_id", "dectec_application_date", "cif", "date_of_birth"   # declined-table names
D_CODES = "dectec_all_reason_decisions"     # per-reason Dectech outcome (A / R / D), same delimiter as the reasons
COL_ALIAS = {"aecb_file_type_revised": "file_type_revised",   # renamed when acard_master_ri was built (2026-11-11)
             "SEX": "gender", "NATIONALITY": "nationality", cust_col: D_CUST}

ri_src = spark.table(RI_CFG["table"])
N_RI_TABLE = ri_src.count()
SELECT_RULE = "UPPER(TRIM(final_decision)) = 'DECLINE' AND UPPER(reason) contains 'INTERNAL SCORE DECLINE'"
dec = (ri_src.filter((F.upper(F.trim(F.col("final_decision"))) == "DECLINE")
                     & F.upper(F.col(REASON_COL)).contains("INTERNAL SCORE DECLINE"))
             .toPandas().reset_index(drop=True))

def norm_text(df):
    """E2b normalisation on text columns: str -> trim -> UPPER, '' = null (common._norm)."""
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object and df[c].map(lambda v: v is None or isinstance(v, str)).all():
            s = common._norm(df[c])
            df[c] = s.astype(object).where(s.notna(), None)
    return df

dec = norm_text(dec)
N_DEC = len(dec)
N_DUP_ID = int(dec[D_ID].duplicated().sum())
lower_cols = {c.lower(): c for c in dec.columns}

def dec_col(c):
    """Declined-table column for a config column name (exact, documented alias, then case-insensitive)."""
    if c in dec.columns:
        return c
    if c in COL_ALIAS and COL_ALIAS[c] in dec.columns:
        return COL_ALIAS[c]
    return lower_cols.get(str(c).lower())

wf = [{"order": 0, "rule": "SELECTION", "column": REASON_COL, "kind": "selection", "condition": SELECT_RULE,
       "declined_column": REASON_COL, "status": "APPLIED", "removed": None, "remaining": N_DEC}]
alive = np.ones(N_DEC, bool)
WF_EXAMPLES, order = {}, 0
for kind in ("exclude", "include"):
    for rule in cfg["population_filters"]["application_time"].get(kind, []):
        order += 1
        c = dec_col(rule["column"])
        row = {"order": order, "rule": rule["name"], "column": rule["column"], "kind": f"application_time.{kind}",
               "condition": f"{rule['op']} {rule['value']}", "declined_column": c}
        if c is None:
            row.update(status="NOT_VERIFIABLE: column not in declined table", removed=None, remaining=int(alive.sum()))
        else:
            hit = common._rule_mask(dec, dict(rule, column=c)).to_numpy()
            fail = (hit if kind == "exclude" else ~hit) & alive
            row.update(status="VERIFIED", removed=int(fail.sum()), remaining=int((alive & ~fail).sum()))
            if fail.any():
                WF_EXAMPLES[rule["name"]] = dec.loc[fail, [D_ID, c]].head(10)
            alive &= ~fail
        wf.append(row)
for kind in ("exclude", "include"):
    for rule in cfg["population_filters"]["performance_related"].get(kind, []):
        order += 1
        wf.append({"order": order, "rule": rule["name"], "column": rule["column"], "kind": f"performance_related.{kind}",
                   "condition": f"{rule['op']} {rule['value']}", "declined_column": None,
                   "status": "NOT_APPLIED: performance-related, does not apply to declined applicants",
                   "removed": None, "remaining": int(alive.sum())})
INTAKE_WF = pd.DataFrame(wf)
WF_REMOVED = int(pd.to_numeric(INTAKE_WF["removed"].iloc[1:], errors="coerce").fillna(0).sum())
N_APP_RULES = sum(len(cfg["population_filters"]["application_time"].get(k, [])) for k in ("exclude", "include"))
N_NOT_VERIFIABLE = int(INTAKE_WF["status"].str.startswith("NOT_VERIFIABLE").sum())
display(INTAKE_WF)
print(f"Declined selected: {N_DEC:,} of {N_RI_TABLE:,} rows in {RI_CFG['table']} | duplicate {D_ID} rows: {N_DUP_ID}")
if WF_REMOVED:
    for k, v in WF_EXAMPLES.items():
        print(f"Rule {k} removed rows, examples:")
        display(v)
    raise AssertionError(f"STOP: verification waterfall removed {WF_REMOVED} declined row(s); see examples above")
print(f"Verification waterfall: 0 rows removed; {N_NOT_VERIFIABLE} of {N_APP_RULES} application-time rules not verifiable")

In [0]:
# ── RI Step 1.3-1.6: approved population, reason mapping (S1 / S2), windows, timing, re-applications ──
# Approved = config.tables.master, every window through common.load_window, sample_status not EXCLUDED
MASTER_TBL = common.table_name(cfg, "master")
m_cols = spark.table(MASTER_TBL).columns
OVERRIDE = set(cfg["column_roles"].get("feature_override", []))
WATCH_SRC = [c for c in cfg["column_roles"]["watch"] if c not in OVERRIDE]
need = [c for c in dict.fromkeys([id_col, "CREDITCARDNO", cust_col, date_col, dec_time_col, status_col, "window", "DATE_OF_BIRTH"]
                                 + WATCH_SRC + list(cfg["columns"]["segments"]) + POOL) if c in m_cols]
master = spark.table(MASTER_TBL).select(*need).toPandas()
appr_all = master.loc[master["window"].isin(list(cfg["windows"])), [cust_col, date_col]].copy()   # any approved record
app = pd.concat([common.load_window(master, w, cfg) for w in cfg["windows"]], ignore_index=True)
n_not_excl = int((master["window"].isin(list(cfg["windows"])) & master[status_col].notna()
                  & master[status_col].ne("EXCLUDED")).sum())
app = app[app[status_col].isin(PSI_STATUSES)].reset_index(drop=True)
assert len(app) == n_not_excl, f"approved P1 ({len(app)}) != sample_status not EXCLUDED ({n_not_excl})"
del master
_d = pd.to_datetime(app[date_col])
app["app_month"], app["app_quarter"] = _d.dt.to_period("M").astype(str), _d.dt.to_period("Q").astype(str)

# Windows for declined: same config.windows as E3, on application_date
cfg_dec = copy.deepcopy(cfg)
cfg_dec["columns"]["date"] = D_DATE
dec = common.assign_window(dec, cfg_dec)
dec["window"] = dec["window"].fillna("OUT_OF_SCOPE").astype(str)
dec["app_month"], dec["app_quarter"] = dec["app_month"].astype(str), dec["app_quarter"].astype(str)
IN_SCOPE = dec["window"].ne("OUT_OF_SCOPE").to_numpy()

# Reason mapping (exact match after split / trim / UPPER)
assert RI_CFG["reason_match"] == "exact", "only reason_match = exact is implemented"
GROUP_OF = {str(v).strip().upper(): g for g, vals in RI_CFG["decision_types"].items() for v in (vals or [])}
EXCL_GROUPS = ["retained_rule", "hard_policy", "not_a_decline"]
SCEN = {"S1": "PENDING reasons do not exclude", "S2": "PENDING reasons exclude"}

def split_list(s):
    return [] if s is None else [t.strip().upper() for t in str(s).split(DELIM)]

long = []
for i, (ds, cs) in enumerate(zip(dec[REASON_COL].map(split_list), dec[D_CODES].map(split_list))):
    for j, r in enumerate(ds):
        if r:
            long.append((i, r, cs[j] if len(cs) == len(ds) and cs[j] else None))
RL = pd.DataFrame(long, columns=["row", "reason", "decision_code"])
RL["group"] = RL["reason"].map(GROUP_OF)
RL.loc[RL["group"].isna() & (RL["decision_code"] == "A"), "group"] = "NEUTRAL_A"   # deviation: approve outcome, not a decline
RL["group"] = RL["group"].fillna("PENDING")                                         # deviation: unmapped, awaiting Credit Policy
FLAG = (pd.crosstab(RL["row"], RL["group"]) > 0).reindex(range(N_DEC), fill_value=False)

def anyg(groups):
    cols = [c for c in groups if c in FLAG.columns]
    return FLAG[cols].any(axis=1).to_numpy() if cols else np.zeros(N_DEC, bool)

HAS_SCORE, HAS_EXCL = anyg(["score_reject"]), anyg(EXCL_GROUPS)
HAS_MANUAL, HAS_PENDING = anyg(["manual_override"]), anyg(["PENDING"])
first_excl = RL[RL["group"].isin(EXCL_GROUPS)].groupby("row")["group"].first().reindex(range(N_DEC))
EXCL_LABEL = ("EXCLUDED_" + first_excl.fillna("").str.upper()).to_numpy()
for s, pending_excludes in (("S1", False), ("S2", True)):
    grp = np.where(HAS_EXCL, EXCL_LABEL,
          np.where(HAS_MANUAL, "MANUAL_OVERRIDE",
          np.where(HAS_PENDING & pending_excludes, "EXCLUDED_PENDING",
          np.where(HAS_SCORE, "INCLUDED", "NO_SCORE_REJECT"))))
    dec[f"RI_GROUP_{s}"] = grp
    dec[f"RI_INCLUDED_{s}"] = np.where(grp == "INCLUDED", "Y", "N")
N_NO_SCORE = int((~HAS_SCORE).sum())

RL["app_month"] = dec["app_month"].to_numpy()[RL["row"].to_numpy()]
REASONS = (RL.groupby(["reason", "group"])
             .agg(n_mentions=("row", "size"), n_apps=("row", "nunique"),
                  decision_codes=("decision_code", lambda x: ",".join(sorted(set(x.dropna())))))
             .reset_index().sort_values("n_apps", ascending=False))
REASONS["status"] = np.where(REASONS["group"].isin(["PENDING", "NEUTRAL_A"]), "UNMAPPED", "MAPPED")
UNMAPPED = REASONS[REASONS["status"] == "UNMAPPED"].reset_index(drop=True)
GROUPS = pd.concat([dec.groupby(["window", f"RI_GROUP_{s}"]).size().unstack(fill_value=0).reset_index().assign(scenario=s)
                    for s in SCEN], ignore_index=True)
BY_MONTH = pd.concat([dec.groupby(["app_month", f"RI_GROUP_{s}"]).size().unstack(fill_value=0).reset_index().assign(scenario=s)
                      for s in SCEN], ignore_index=True)
REASON_MONTH = RL.groupby(["app_month", "group"])["row"].nunique().unstack(fill_value=0).reset_index()

# Derived features of declined applicants as at application_date
_ad, _dob = pd.to_datetime(dec[D_DATE]).dt.normalize(), pd.to_datetime(dec[D_DOB]).dt.normalize()
age_re = np.floor((_ad - _dob).dt.days / 365.25)
age_tab = pd.to_numeric(dec["AGE"], errors="coerce")
N_AGE_MISMATCH = int(((age_re != age_tab) & age_re.notna() & age_tab.notna()).sum() + (age_re.isna() != age_tab.isna()).sum())
dec["AGE"] = age_re

# Timing: approved APPROVEDATE - APPRECDDATE; declined application_date - AECB report date
def describe_days(days, by, label):
    d = (pd.DataFrame({"by": by, "days": days}).groupby("by")["days"]
           .describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).reset_index())
    d.insert(0, "measure", label)
    return d
gap_a = (pd.to_datetime(app[dec_time_col]) - pd.to_datetime(app[date_col])).dt.days
gap_d = (_ad - pd.to_datetime(dec["aecb_CreatedON"])).dt.days
TIMING = pd.concat([
    describe_days(gap_a, app["window"], f"approved P1: {dec_time_col} - {date_col} (days)"),
    describe_days(gap_a, np.repeat("ALL", len(app)), f"approved P1: {dec_time_col} - {date_col} (days)"),
    describe_days(gap_d, dec["window"], f"declined: {D_DATE} - aecb_CreatedON (days)"),
    describe_days(gap_d, np.repeat("ALL", N_DEC), f"declined: {D_DATE} - aecb_CreatedON (days)")], ignore_index=True)

# Re-applications: declined customers approved later (any approved record in config.tables.master)
appr_all["_c"] = common._norm(appr_all[cust_col]).astype(object)
appr_all["_d"] = pd.to_datetime(appr_all[date_col])
dc = dec.loc[dec[D_CUST].notna(), [D_CUST, D_DATE]].reset_index()
mm = dc.merge(appr_all.dropna(subset=["_c"])[["_c", "_d"]], left_on=D_CUST, right_on="_c", how="inner")
later = set(mm.loc[mm["_d"] > pd.to_datetime(mm[D_DATE]), "index"])
dec["reapplied_later_approved"] = np.where(dec.index.isin(later), "Y", "N")
N_WITH_CUST = int(dec[D_CUST].notna().sum())
N_REAPPLIED = int((dec["reapplied_later_approved"] == "Y").sum())

print(f"Approved P1: {len(app):,} | declined: {N_DEC:,} (in scope {int(IN_SCOPE.sum()):,}, OUT_OF_SCOPE {int((~IN_SCOPE).sum()):,})")
print(f"Reasons: {len(REASONS)} distinct, {len(UNMAPPED)} unmapped | apps without score_reject: {N_NO_SCORE}")
for s in SCEN:
    print(f"{s} ({SCEN[s]}): RI_INCLUDED = Y {int((dec[f'RI_INCLUDED_{s}'] == 'Y').sum()):,} | "
          f"groups {dec[f'RI_GROUP_{s}'].value_counts().to_dict()}")
print(f"AGE recomputed at application_date: {N_AGE_MISMATCH} difference(s) vs table AGE")
print(f"Declined with {D_CUST}: {N_WITH_CUST:,} | later approved: {N_REAPPLIED:,}")
display(dec.groupby(["window"]).size().rename("declined").reset_index())

In [0]:
# ── RI Step 2: PSI / CSI declined vs approved on the L1 bins, missing rates, policy truncation, through-the-door ──
TRUNC_APP_MAX, TRUNC_DEC_MIN = 0.05, 0.20        # module request 2026-11-11 (not in config)
PSI_G, PSI_R = cfg["funnel"]["L3"]["psi_green"], cfg["funnel"]["L3"]["psi_red"]
WIN3 = ["DEV", "OOT", "FWD"]                      # windows that contain declined applications

def bin_index(f, s):
    """L1 bin index; missing and each special value get their own bin (OptBinning 'empirical')."""
    v = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    return np.asarray(L1[f].transform(v, metric="indices", metric_missing="empirical", metric_special="empirical"))

def bin_labels(f):
    t = L1[f].binning_table.build()
    return [str(b) for b in t["Bin"].iloc[:-1]]      # last row = Totals; order = bins, Special, Missing

def psi(a, b):
    return common.psi_from_bins(pd.Series(a), pd.Series(b), EPS) if len(a) and len(b) else np.nan

def psi_status(v):
    return None if pd.isna(v) else ("GREEN" if v <= PSI_G else ("YELLOW" if v <= PSI_R else "RED"))

def models_of(f):
    return ",".join(m for m in RI_MODELS if f in MODEL_FEATS[m])

APP_W, DEC_W = app["window"].astype(str).to_numpy(), dec["window"].to_numpy()
APP_Q, DEC_Q = app["app_quarter"].to_numpy(), dec["app_quarter"].to_numpy()
APP_M, DEC_M = app["app_month"].to_numpy(), dec["app_month"].to_numpy()
APP_IN = np.isin(APP_W, WIN3)
BA = {f: bin_index(f, app[f]) for f in POOL}
BD = {f: bin_index(f, dec[f]) for f in POOL}
LABELS = {f: bin_labels(f) for f in POOL}
NA_A = {f: pd.to_numeric(app[f], errors="coerce").isna().to_numpy() for f in POOL}
NA_D = {f: pd.to_numeric(dec[f], errors="coerce").isna().to_numpy() for f in POOL}
DSETS = {"ALL_DECLINED": IN_SCOPE,
         **{f"{s}_INCLUDED": IN_SCOPE & (dec[f"RI_INCLUDED_{s}"] == "Y").to_numpy() for s in SCEN}}

# 2.1 PSI by window (DEV, OOT as specified; FWD for diagnosis) and by quarter; baseline = approved
rows = []
for sname, smask in DSETS.items():
    for w in WIN3:
        am, dm = APP_W == w, smask & (DEC_W == w)
        for f in POOL:
            v = psi(BA[f][am], BD[f][dm])
            rows.append({"declined_set": sname, "window": w, "use": "diagnosis only" if w == "FWD" else "spec",
                         "feature": f, "models": models_of(f), "n_approved": int(am.sum()), "n_declined": int(dm.sum()),
                         "psi": v, "status": psi_status(v)})
PSI_WIN = pd.DataFrame(rows)
QUARTERS = sorted(set(DEC_Q[IN_SCOPE]))
rows = []
for sname, smask in DSETS.items():
    for q in QUARTERS:
        am, dm = APP_IN & (APP_Q == q), smask & (DEC_Q == q)
        for f in POOL:
            v = psi(BA[f][am], BD[f][dm])
            rows.append({"declined_set": sname, "quarter": q, "feature": f, "models": models_of(f),
                         "n_approved": int(am.sum()), "n_declined": int(dm.sum()), "psi": v, "status": psi_status(v)})
PSI_Q = pd.DataFrame(rows)

# 2.2 Missing rate by group and quarter (declined = all in-scope declines)
rows = []
for q in sorted(set(DEC_Q[IN_SCOPE]) | set(APP_Q[APP_IN])):
    am, dm = APP_IN & (APP_Q == q), IN_SCOPE & (DEC_Q == q)
    for f in POOL:
        rows.append({"quarter": q, "feature": f, "models": models_of(f), "n_approved": int(am.sum()), "n_declined": int(dm.sum()),
                     "missing_approved": NA_A[f][am].mean() if am.any() else np.nan,
                     "missing_declined": NA_D[f][dm].mean() if dm.any() else np.nan})
MISSING = pd.DataFrame(rows)
MISSING["diff_pp"] = (MISSING["missing_declined"] - MISSING["missing_approved"]) * 100

# 2.3 Policy-truncated bins: < 5% of approved and > 20% of declined
rows = []
for sname, smask in DSETS.items():
    for w in ["DEV", "OOT", "ALL"]:
        am = APP_IN if w == "ALL" else (APP_W == w)
        dm = smask if w == "ALL" else smask & (DEC_W == w)
        if not dm.any():
            continue
        for f in POOL:
            sa = pd.Series(BA[f][am]).value_counts(normalize=True)
            sd = pd.Series(BD[f][dm]).value_counts(normalize=True)
            for b in sorted(set(sa.index) | set(sd.index)):
                a_, d_ = float(sa.get(b, 0.0)), float(sd.get(b, 0.0))
                rows.append({"declined_set": sname, "window": w, "feature": f, "models": models_of(f), "bin_index": int(b),
                             "bin": LABELS[f][b] if b < len(LABELS[f]) else str(b), "approved_share": a_,
                             "declined_share": d_, "truncated": bool(a_ < TRUNC_APP_MAX and d_ > TRUNC_DEC_MIN)})
POLICY_TRUNC = pd.DataFrame(rows)
TRUNC_FEATS = {f"{k[0]} / {k[1]}": sorted(v["feature"].unique())
               for k, v in POLICY_TRUNC[POLICY_TRUNC["truncated"]].groupby(["declined_set", "window"])}

# 2.4 Through-the-door: monthly counts and PSI of all applicants (approved + declined) vs approved (diagnosis only)
rows = []
for mo in sorted(set(APP_M[APP_IN]) | set(DEC_M[IN_SCOPE])):
    na, nd = int((APP_IN & (APP_M == mo)).sum()), int((IN_SCOPE & (DEC_M == mo)).sum())
    r = {"app_month": mo, "approved": na, "declined_isd": nd, "approval_rate": na / (na + nd) if na + nd else np.nan}
    for s in SCEN:
        r[f"declined_{s}_included"] = int((DSETS[f"{s}_INCLUDED"] & (DEC_M == mo)).sum())
    rows.append(r)
TTD_MONTHLY = pd.DataFrame(rows)
rows = []
for w in WIN3:
    am, dm = APP_W == w, IN_SCOPE & (DEC_W == w)
    for f in POOL:
        v = psi(BA[f][am], np.concatenate([BA[f][am], BD[f][dm]]))
        rows.append({"window": w, "feature": f, "models": models_of(f), "n_approved": int(am.sum()),
                     "n_all": int(am.sum() + dm.sum()), "psi_all_vs_approved": v, "status": psi_status(v)})
TTD_PSI = pd.DataFrame(rows)

print("PSI declined vs approved (ALL_DECLINED), status counts by window:")
display(PSI_WIN[PSI_WIN["declined_set"] == "ALL_DECLINED"].groupby(["window", "status"]).size().unstack(fill_value=0))
print("Policy-truncated features:", TRUNC_FEATS)

In [0]:
# ── RI Step 3.1: score approved and declined with the E13 scoring code (raw -> Platt -> PDO score) ──
def ri_score_raw(df):
    """E13 score_raw (s05_oot, 'Score with the locked models'), models A and C."""
    XA = df[A_FEATS].apply(pd.to_numeric, errors="coerce")
    pa = A_LR.predict_proba(np.asarray(A_BP.transform(XA, metric="woe", **WOE_KW), dtype=float))[:, 1]
    XR = common.prepare_raw(df, POOL, RI_FDICT)
    assert list(XR.columns) == BC_COLS, "prepare_raw columns differ from the locked B/C columns"
    pc = C_BST.predict(XR)
    return {"A": pa, "C": pc}

def ri_score(df):
    raw = ri_score_raw(df)
    out = {}
    for m in RI_MODELS:
        assert np.isfinite(raw[m]).all() and ((raw[m] > 0) & (raw[m] < 1)).all(), f"{m}: raw PD outside (0, 1)"
        pd_cal = common.apply_platt(raw[m], PLATT[m])
        out[m] = {"raw": raw[m], "pd": pd_cal, "score": common.pd_to_score(pd_cal, cfg)}
    return out

SA, SD = ri_score(app), ri_score(dec)

# DEV cut-points: deciles of each model's calibrated PD on the frozen DEV GOOD+BAD population (as E13)
frozen_ids = set(pd.read_parquet(FILES["folds/r_folds.parquet"], columns=["app_id"])["app_id"].astype(str))
DEV_GB = (APP_W == "DEV") & app[status_col].isin(LABELLED).to_numpy() & app[id_col].astype(str).isin(frozen_ids).to_numpy()
_exp = cfg["expected_counts"]["DEV"]
assert DEV_GB.sum() == _exp["records"] and int((app.loc[DEV_GB, status_col] == "BAD").sum()) == _exp["bads"], \
    f"DEV GOOD+BAD {int(DEV_GB.sum())} != expected {_exp}"

def edges(p, q):
    return np.unique(np.quantile(p, np.linspace(0, 1, q + 1)[1:-1]))

def band(p, e):
    return np.searchsorted(e, p, side="right") + 1      # 1 = lowest PD (best) ... 10 = highest PD

EDGES = {m: edges(SA[m]["pd"][DEV_GB], 10) for m in RI_MODELS}

# Spot check: approved OOT scores reproduce the stored E13 scores (config.tables.oot_predictions)
OOT_PRED_TBL = common.table_name(cfg, "oot_predictions")
SPOT_KEY = [id_col, "CREDITCARDNO"]       # CRNNO alone is not row-unique (one row per card); the pair is
stored = spark.table(OOT_PRED_TBL).select(*SPOT_KEY, *[f"prob_{m}_cal" for m in RI_MODELS],
                                          *[f"score_{m}" for m in RI_MODELS]).toPandas()
for k in SPOT_KEY:
    stored[k] = stored[k].astype(str)
mine = pd.DataFrame({**{k: app[k].astype(str).to_numpy() for k in SPOT_KEY},
                     **{f"my_pd_{m}": SA[m]["pd"] for m in RI_MODELS},
                     **{f"my_score_{m}": SA[m]["score"] for m in RI_MODELS}})[APP_W == "OOT"]
assert not stored.duplicated(SPOT_KEY).any() and not mine.duplicated(SPOT_KEY).any(), "duplicate keys in the spot-check join"
jj = stored.merge(mine, on=SPOT_KEY, how="inner")
assert len(jj) >= 1000, f"only {len(jj)} OOT rows match {OOT_PRED_TBL}"
spot = jj.sample(n=1000, random_state=SEED)
SPOT = {m: {"n": len(spot),
            "max_abs_pd_diff": float(np.abs(spot[f"prob_{m}_cal"] - spot[f"my_pd_{m}"]).max()),
            "score_mismatches": int((spot[f"score_{m}"].astype(int) != spot[f"my_score_{m}"].astype(int)).sum())}
        for m in RI_MODELS}
SPOT_OK = all(v["max_abs_pd_diff"] <= 1e-9 and v["score_mismatches"] == 0 for v in SPOT.values())
print(f"OOT rows matched to {OOT_PRED_TBL}: {len(jj):,}; spot check on 1,000: {SPOT} -> {'OK' if SPOT_OK else 'FAIL'}")
print({m: [round(x, 5) for x in EDGES[m]] for m in RI_MODELS}, "<- DEV decile PD cut-points")

In [0]:
# ── RI Step 3.2-3.4 (RI_INCLUDED = Y only, per scenario): deciles, extrapolation, swap set ──
EXT_P_LO, EXT_P_HI, EXT_RARE = 1, 99, 0.01        # module request 2026-11-11 (not in config)
PROPOSED = cfg.get("strategy", {}).get("proposed_cutoff")
INC = {s: DSETS[f"{s}_INCLUDED"] for s in SCEN}
BAND_A = {m: band(SA[m]["pd"], EDGES[m]) for m in RI_MODELS}
BAND_D = {m: band(SD[m]["pd"], EDGES[m]) for m in RI_MODELS}

# 3.2 Declined across the approved score deciles (DEV cut-points)
rows = []
for m in RI_MODELS:
    nd = len(EDGES[m]) + 1
    for w in ["ALL"] + WIN3:
        am = APP_IN if w == "ALL" else (APP_W == w)
        for d in range(1, nd + 1):
            r = {"model": m, "window": w, "decile": d,
                 "dev_pd_upper": float(EDGES[m][d - 1]) if d <= len(EDGES[m]) else 1.0,
                 "approved_n": int((am & (BAND_A[m] == d)).sum()),
                 "approved_share": float((BAND_A[m][am] == d).mean()) if am.any() else np.nan}
            for s in SCEN:
                dm = INC[s] if w == "ALL" else INC[s] & (DEC_W == w)
                r[f"declined_{s}_n"] = int((dm & (BAND_D[m] == d)).sum())
                r[f"declined_{s}_share"] = float((BAND_D[m][dm] == d).mean()) if dm.any() else np.nan
            rows.append(r)
SCORE_DIST = pd.DataFrame(rows)
rows = []
for m in RI_MODELS:
    for w in ["ALL"] + WIN3:
        am = APP_IN if w == "ALL" else (APP_W == w)
        rows.append({"model": m, "window": w, "group": "approved", "n": int(am.sum()), "mean_pd": SA[m]["pd"][am].mean(),
                     "median_score": float(np.median(SA[m]["score"][am])) if am.any() else np.nan})
        for s in SCEN:
            dm = INC[s] if w == "ALL" else INC[s] & (DEC_W == w)
            rows.append({"model": m, "window": w, "group": f"declined_{s}", "n": int(dm.sum()),
                         "mean_pd": SD[m]["pd"][dm].mean() if dm.any() else np.nan,
                         "median_score": float(np.median(SD[m]["score"][dm])) if dm.any() else np.nan})
SCORE_SUM = pd.DataFrame(rows)

# 3.3 Extrapolation vs approved DEV P1: outside 1st-99th percentile, or in an L1 bin holding < 1% of approved
REF = APP_W == "DEV"
EXT = {}
for f in POOL:
    v = pd.to_numeric(app.loc[REF, f], errors="coerce").dropna().to_numpy()
    lo, hi = np.percentile(v, [EXT_P_LO, EXT_P_HI])
    sh = pd.Series(BA[f][REF]).value_counts(normalize=True)
    rare = [int(b) for b in sh.index[sh < EXT_RARE]]
    vd = pd.to_numeric(dec[f], errors="coerce").to_numpy(dtype=float)
    outside = (vd < lo) | (vd > hi)                         # missing -> not outside
    rare_d = np.isin(BD[f], rare) | ~np.isin(BD[f], sh.index.to_numpy())
    EXT[f] = {"p01": float(lo), "p99": float(hi), "rare_bins": rare, "outside": outside, "rare": rare_d}
rows, EXT_ANY = [], {}
for s in SCEN:
    dm = INC[s]
    for m in RI_MODELS:
        flag_any = np.zeros(N_DEC, bool)
        for f in MODEL_FEATS[m]:
            e = EXT[f]
            either = e["outside"] | e["rare"]
            flag_any |= either
            rows.append({"scenario": s, "model": m, "feature": f, "n_declined": int(dm.sum()),
                         "approved_dev_p01": e["p01"], "approved_dev_p99": e["p99"],
                         "rare_bins": ",".join(LABELS[f][b] if b < len(LABELS[f]) else str(b) for b in e["rare_bins"]),
                         "share_outside_p01_p99": float(e["outside"][dm].mean()) if dm.any() else np.nan,
                         "share_in_rare_bin": float(e["rare"][dm].mean()) if dm.any() else np.nan,
                         "share_either": float(either[dm].mean()) if dm.any() else np.nan})
        EXT_ANY[(s, m)] = float(flag_any[dm].mean()) if dm.any() else np.nan
        rows.append({"scenario": s, "model": m, "feature": "ANY (>= 1 model feature)", "n_declined": int(dm.sum()),
                     "share_either": EXT_ANY[(s, m)]})
EXTRAP = pd.DataFrame(rows)

# 3.4 Swap set at strategy.proposed_cutoff, or at each approved DEV score decile when it is null
CUTOFFS = {m: ([PROPOSED] if PROPOSED is not None else sorted(set(common.pd_to_score(EDGES[m], cfg).tolist())))
           for m in RI_MODELS}
LAB = app[status_col].isin(LABELLED).to_numpy() & np.isin(APP_W, ["DEV", "OOT"])   # labels: approved GOOD/BAD only
BAD = (app[status_col] == "BAD").to_numpy()
rows = []
for s in SCEN:
    months = sorted(set(APP_M[APP_IN]) | set(DEC_M[INC[s]]))
    levels = ([("ALL", "ALL", APP_IN, INC[s])] + [("window", w, APP_W == w, INC[s] & (DEC_W == w)) for w in WIN3]
              + [("month", mo, APP_IN & (APP_M == mo), INC[s] & (DEC_M == mo)) for mo in months])
    for m in RI_MODELS:
        for c in CUTOFFS[m]:
            ap_ok, de_ok = SA[m]["score"] >= c, SD[m]["score"] >= c
            for lt, lv, am, dm in levels:
                n_a, n_d = int(am.sum()), int(dm.sum())
                si, bd_ = int((dm & de_ok).sum()), int((dm & ~de_ok).sum())
                so, ba_ = int((am & ~ap_ok).sum()), int((am & ap_ok).sum())
                sol = am & ~ap_ok & LAB
                rows.append({"scenario": s, "model": m, "cutoff": int(c), "level_type": lt, "level": lv,
                             "n_approved": n_a, "n_declined": n_d, "swap_in": si, "swap_out": so,
                             "both_approve": ba_, "both_decline": bd_,
                             "reconciles": bool(si + bd_ == n_d and so + ba_ == n_a and si + so + ba_ + bd_ == n_a + n_d),
                             "swap_out_labelled_n": int(sol.sum()), "swap_out_bads": int((sol & BAD).sum()),
                             "swap_out_bad_rate_observed": float(BAD[sol].mean()) if sol.any() else np.nan,
                             "swap_in_mean_pd_expected": float(SD[m]["pd"][dm & de_ok].mean()) if si else np.nan})
SWAP = pd.DataFrame(rows)
_cnt = ["n_approved", "n_declined", "swap_in", "swap_out", "both_approve", "both_decline"]
_k = ["scenario", "model", "cutoff"]
_tot = SWAP[SWAP["level_type"] == "ALL"].set_index(_k)[_cnt].sort_index()
SWAP_WIN_OK = bool((SWAP[SWAP["level_type"] == "window"].groupby(_k)[_cnt].sum().sort_index() == _tot).all().all())
SWAP_MON_OK = bool((SWAP[SWAP["level_type"] == "month"].groupby(_k)[_cnt].sum().sort_index() == _tot).all().all())
SWAP_OK = bool(SWAP["reconciles"].all()) and SWAP_WIN_OK and SWAP_MON_OK
print(f"Swap rows: {len(SWAP):,} | reconcile per row: {bool(SWAP['reconciles'].all())} | windows sum to total: {SWAP_WIN_OK} "
      f"| months sum to total: {SWAP_MON_OK}")
print("Extrapolation, share of declined with >= 1 flagged feature:", {f"{k[0]}/{k[1]}": round(v, 4) for k, v in EXT_ANY.items()})
display(SCORE_SUM[SCORE_SUM["window"] == "ALL"])

In [0]:
# ── RI Step 3.5: Steps 2 and 3 by config.columns.segments and every WATCH variable ──
SEG_MIN_N = 100     # report-only grouping: values with fewer approved + declined records are pooled as OTHER
SEG_COLS = [c for c in cfg["columns"]["segments"] if c in app.columns and dec_col(c)]
SEG_MISSING = [c for c in cfg["columns"]["segments"] if c not in SEG_COLS]
WATCH_COLS = [c for c in WATCH_SRC if c in app.columns and dec_col(c)]
AGE_COL = next((c for c in cfg["column_roles"]["watch"] if c in OVERRIDE), None)

def lab(s):
    x = common._norm(s)
    return x.astype(object).where(x.notna(), "MISSING").to_numpy()

SEGVALS = {}
for c in SEG_COLS + WATCH_COLS:
    SEGVALS[c] = (lab(app[c]), lab(dec[dec_col(c)]), "segment" if c in SEG_COLS else "watch")
AGE_EDGES = []
if AGE_COL:
    AGE_EDGES = np.unique(np.quantile(pd.to_numeric(app.loc[DEV_GB, AGE_COL], errors="coerce").dropna(), [0.2, 0.4, 0.6, 0.8]))
    def age_band(s):
        v = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
        names = ([f"<{AGE_EDGES[0]:g}"] + [f"[{a:g},{b:g})" for a, b in zip(AGE_EDGES[:-1], AGE_EDGES[1:])]
                 + [f">={AGE_EDGES[-1]:g}"])
        idx = np.minimum(np.searchsorted(AGE_EDGES, v, side="right"), len(names) - 1)
        return np.where(np.isnan(v), "MISSING", np.array(names, dtype=object)[idx])
    SEGVALS["age_band"] = (age_band(app[AGE_COL]), age_band(dec["AGE"]), "watch")

rows = []
for var, (va, vd, typ) in SEGVALS.items():
    cnt = pd.Series(np.concatenate([va[APP_IN], vd[IN_SCOPE]])).value_counts()
    small = set(cnt.index[cnt < SEG_MIN_N])
    va = np.where(np.isin(va, list(small)), "OTHER", va)
    vd = np.where(np.isin(vd, list(small)), "OTHER", vd)
    seen = set(va[APP_IN])
    rows.append({"analysis": "coding_check", "variable": var, "type": typ,
                 "metric": "share of in-scope declined whose value is not seen among approved",
                 "value_num": float(np.mean(~np.isin(vd[IN_SCOPE], list(seen)))) if IN_SCOPE.any() else np.nan,
                 "value": f"{len(seen)} approved values / {len(set(vd[IN_SCOPE]))} declined values"})
    for val in sorted(set(va[APP_IN]) | set(vd[IN_SCOPE])):
        am, dm = APP_IN & (va == val), IN_SCOPE & (vd == val)
        base = {"variable": var, "type": typ, "value": val, "n_approved": int(am.sum()), "n_declined": int(dm.sum())}
        for f in POOL:                                   # Step 2 by segment: PSI declined vs approved
            v = psi(BA[f][am], BD[f][dm])
            rows.append({**base, "analysis": "psi", "feature": f, "models": models_of(f), "metric": "psi",
                         "value_num": v, "status": psi_status(v)})
        for s in SCEN:                                   # Step 3 by segment
            dms = INC[s] & (vd == val)
            for m in RI_MODELS:
                for d in range(1, len(EDGES[m]) + 2):
                    rows.append({**base, "n_declined": int(dms.sum()), "analysis": "decile_share", "scenario": s,
                                 "model": m, "decile": d, "metric": "declined share in DEV decile",
                                 "value_num": float((BAND_D[m][dms] == d).mean()) if dms.any() else np.nan})
                for c in CUTOFFS[m]:
                    si = int((dms & (SD[m]["score"] >= c)).sum())
                    so = int((am & (SA[m]["score"] < c)).sum())
                    rows.append({**base, "n_declined": int(dms.sum()), "analysis": "swap", "scenario": s, "model": m,
                                 "cutoff": int(c), "metric": "swap_in", "value_num": si})
                    rows.append({**base, "n_declined": int(dms.sum()), "analysis": "swap", "scenario": s, "model": m,
                                 "cutoff": int(c), "metric": "swap_out", "value_num": so})
SEGMENTS = pd.DataFrame(rows)
print(f"Segments: {SEG_COLS} (missing: {SEG_MISSING}) | watch: {WATCH_COLS} + age_band {list(AGE_EDGES)} | rows {len(SEGMENTS):,}")
display(SEGMENTS[SEGMENTS["analysis"] == "coding_check"])

In [0]:
# ── RI: open questions / deviations, summary, report workbook (all under the RI output folder) ──
RI_RUN_TIME = datetime.now().isoformat(timespec="seconds")
OQ_LINES = [
    "# Reject analysis (s09 RI section): open questions and deviations", "",
    f"Run {RI_RUN_TIME} | registry {REG['version']} | git {GIT_COMMIT[:12]} | config {CONFIG_HASH[:12]}", "",
    "1. Unmapped decline reasons. Report-only deviation approved by the model owner on 2026-11-11: "
    "`unclassified_action: stop` is not applied. Unmapped reasons are PENDING; Steps 2-3 run under S1 (PENDING does not "
    "exclude) and S2 (PENDING excludes). Reasons to be placed in `decision_types` by Credit Policy:",
    *[f"   - {r.reason} [{r.group}, Dectech {r.decision_codes}] ({r.n_apps} apps)" for r in UNMAPPED.itertuples()],
    "2. Reasons with Dectech outcome A (e.g. SYSTEM APPROVE) are not decline reasons. With exclude_if_any_excluded they "
    "cannot go to not_a_decline without removing the pure score declines, so they are NEUTRAL in both scenarios. "
    "A config group for neutral reasons is needed; spec Chapter 12 does not cover this.",
    f"3. Application-time population filters: {N_NOT_VERIFIABLE} of {N_APP_RULES} rules are NOT_VERIFIABLE (columns not in "
    f"{RI_CFG['table']}); owner decision 2026-11-11. THIN_FILE_ONLY is verified through `file_type_revised` "
    "(= aecb_file_type_revised, renamed when the table was built).",
    f"4. {RI_CFG['table']} is not pre-filtered as `reject_inference.prefiltered` states ({N_RI_TABLE:,} rows). This section "
    f"selects {SELECT_RULE} ({N_DEC:,} rows). {N_DUP_ID} duplicate application_id rows are kept (spec silent).",
    f"5. Model C is analysed next to the registry selection ({SELECTED}, status {SEL_STATUS}) at the owner's request.",
    "6. The E13 scoring code is inline in s05_oot (no shared scoring module); score_raw is re-implemented verbatim for A "
    "and C, and equivalence is shown by the 1,000-row spot check against config.tables.oot_predictions.",
    "7. Thresholds taken from the module request, not config: policy truncation (< 5% approved, > 20% declined), "
    "extrapolation (approved DEV 1st-99th percentile; L1 bins < 1% of approved DEV), segment pooling (< 100 records).",
    "8. Through-the-door: declined = internal-score declines only, so approval rate = approved / (approved + ISD declines).",
    "9. WATCH coding: approved SEX / NATIONALITY and declined gender / nationality may be coded differently; see the "
    "segments sheet (coding_check).",
    f"10. Declined derived features as at application_date: AGE = floor(days(application_date - date_of_birth) / 365.25) "
    f"({N_AGE_MISMATCH} differences vs table AGE). Approved AGE in the master table is as at APPROVEDATE. Bureau features "
    "come from the AECB report attached to the application (gap to aecb_CreatedON in timing_gap).",
    f"11. config.columns.segments not available: {SEG_MISSING}.",
    f"12. Customer id: {N_WITH_CUST:,} of {N_DEC:,} declined rows have `cif`; reapplied_later_approved covers only these.",
    "13. No MLflow run is logged for this section (s09 does not log to MLflow).",
]
OQ_TEXT = "\n".join(OQ_LINES) + "\n"
_tmp = os.path.join(RI_TMP, "open_questions.md")
with open(_tmp, "w") as fh:
    fh.write(OQ_TEXT)
shutil.copyfile(_tmp, ri_out(RI_OQ))

# Summary sheet
S = []
def add(section, metric, value, scenario=None, model=None):
    S.append({"section": section, "scenario": scenario, "model": model, "metric": metric,
              "value": value if isinstance(value, (int, float, np.integer, np.floating)) else str(value)})
tot_a, tot_d = int(APP_IN.sum()), int(IN_SCOPE.sum())
add("intake", "declined selected / table rows", f"{N_DEC:,} / {N_RI_TABLE:,}")
add("intake", "verification waterfall rows removed", WF_REMOVED)
add("intake", "application-time rules not verifiable", f"{N_NOT_VERIFIABLE} of {N_APP_RULES}")
add("approval", "approved P1 (DEV+OOT+FWD)", tot_a)
add("approval", "internal-score declines in scope", tot_d)
add("approval", "approval rate = approved / (approved + ISD declines)", tot_a / (tot_a + tot_d))
add("reason_groups", "unmapped reasons (PENDING + NEUTRAL_A)", len(UNMAPPED))
for s in SCEN:
    for g_, n_ in dec.loc[IN_SCOPE, f"RI_GROUP_{s}"].value_counts().items():
        add("reason_groups", g_, int(n_), s)
for s in SCEN:
    for m in RI_MODELS:
        sd_ = SCORE_DIST[(SCORE_DIST["model"] == m) & (SCORE_DIST["window"] == "ALL")]
        for r in sd_.itertuples():
            add("declined_by_score_band", f"DEV decile {r.decile} share", getattr(r, f"declined_{s}_share"), s, m)
        sw = SWAP[(SWAP["scenario"] == s) & (SWAP["model"] == m) & (SWAP["level_type"] == "ALL")]
        for r in sw.itertuples():
            add("swap", f"cut-off {r.cutoff}: swap-in / swap-out", f"{r.swap_in} / {r.swap_out}", s, m)
        add("extrapolation", "share of declined with >= 1 extrapolated model feature", EXT_ANY[(s, m)], s, m)
for k, v in TRUNC_FEATS.items():
    add("policy_truncated", k, ", ".join(v))
if not TRUNC_FEATS:
    add("policy_truncated", "features", "none")
for m, v in SPOT.items():
    add("spot_check", "max |PD diff| / score mismatches (1,000 OOT rows)", f"{v['max_abs_pd_diff']:.2e} / {v['score_mismatches']}", None, m)
SUMMARY_RI = pd.DataFrame(S)

META_RI = pd.DataFrame([{
    "module": "S09_RI_reject_analysis", "run_time": RI_RUN_TIME, "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH,
    "branch": RUN_CTX.get("branch"), "registry_version": REG["version"], "registry_sha256": REG_SHA0,
    "selected_model": SELECTED, "selected_status": SEL_STATUS, "models_analysed": ",".join(RI_MODELS),
    "calibration_file": CAL_FILE["path"], "declined_table": RI_CFG["table"], "declined_selection": SELECT_RULE,
    "approved_table": MASTER_TBL, "approved_population": "sample_status not EXCLUDED (config.psi_statuses)",
    "spot_check_table": OOT_PRED_TBL, "scenarios": json.dumps(SCEN), "reason_delimiter": DELIM,
    "open_questions": RI_OQ}])

def write_blocks(w, sheet, blocks):
    r = 0
    for title, df in blocks:
        pd.DataFrame({title: []}).to_excel(w, sheet_name=sheet, startrow=r, index=False)
        df.to_excel(w, sheet_name=sheet, startrow=r + 1, index=False)
        r += len(df) + 4

_tmp_xl = os.path.join(RI_TMP, "ri_reject_analysis.xlsx")
with pd.ExcelWriter(_tmp_xl, engine="openpyxl") as w:
    META_RI.T.reset_index().rename(columns={"index": "field", 0: "value"}).to_excel(w, sheet_name="meta", index=False)
    SUMMARY_RI.to_excel(w, sheet_name="summary", index=False)
    INTAKE_WF.to_excel(w, sheet_name="intake_waterfall", index=False)
    write_blocks(w, "reasons", [("Reasons (split, trim, UPPER)", REASONS),
                                ("RI group by window and scenario", GROUPS),
                                ("RI group by month and scenario", BY_MONTH),
                                ("Applications per reason group by month", REASON_MONTH)])
    TIMING.to_excel(w, sheet_name="timing_gap", index=False)
    PSI_WIN.to_excel(w, sheet_name="psi_by_window", index=False)
    PSI_Q.to_excel(w, sheet_name="psi_by_quarter", index=False)
    MISSING.to_excel(w, sheet_name="missing_rates", index=False)
    POLICY_TRUNC.to_excel(w, sheet_name="policy_truncated", index=False)
    write_blocks(w, "through_the_door", [("Monthly approved, declined, approval rate", TTD_MONTHLY),
                                         ("PSI all applicants vs approved (diagnosis only)", TTD_PSI)])
    write_blocks(w, "score_distribution", [("Declined across approved DEV deciles", SCORE_DIST),
                                           ("Mean PD and median score", SCORE_SUM)])
    EXTRAP.to_excel(w, sheet_name="extrapolation", index=False)
    SWAP.to_excel(w, sheet_name="swap_set", index=False)
    SEGMENTS.to_excel(w, sheet_name="segments", index=False)
shutil.copyfile(_tmp_xl, ri_out(RI_XLSX))
print(f"Saved {RI_XLSX}\nSaved {RI_OQ}")
display(SUMMARY_RI)

In [0]:
# ── RI acceptance (asserts) and run summary ──
REG_SHA1 = sha256_file(REGISTRY_PATH)
HASH1 = {k: sha256_file(p) for k, p in FILES.items()}
CAL_SHA1 = sha256_file(CAL_FILE["path"])
REG_OK = REG_SHA1 == REG_SHA0 and HASH1 == HASH0 and all(HASH1[k] == v["sha256"] for k, v in REG["files"].items()) \
         and CAL_SHA1 == CAL_SHA0
SNAP1 = tree_snapshot(ARTIFACTS, RI_DIR)
CHANGED_OUTSIDE = sorted(p for p in set(SNAP0) | set(SNAP1) if SNAP0.get(p) != SNAP1.get(p))
WRITE_OK = not CHANGED_OUTSIDE and all(p.startswith(RI_DIR + os.sep) for p in RI_WRITTEN)
UNMAPPED_DOCUMENTED = all(r in OQ_TEXT for r in UNMAPPED["reason"])

ri_summary = {
    "module": "S09_RI_reject_analysis", "run_time": datetime.now().isoformat(timespec="seconds"),
    "git_commit": GIT_COMMIT, "config_hash": CONFIG_HASH, "registry_version": REG["version"], "registry_sha256": REG_SHA0,
    "selected_model": SELECTED, "selected_status": SEL_STATUS, "models": RI_MODELS,
    "rows": {"ri_table": N_RI_TABLE, "declined_selected": N_DEC, "declined_in_scope": int(IN_SCOPE.sum()),
             "approved_p1": len(app), **{f"ri_included_{s}": int(INC[s].sum()) for s in SCEN}},
    "checks": {"registry_and_files_unchanged": REG_OK, "no_write_outside_output_folder": WRITE_OK,
               "waterfall_rows_removed": WF_REMOVED, "rules_not_verifiable": N_NOT_VERIFIABLE,
               "unmapped_reasons": len(UNMAPPED), "unmapped_documented_as_deviation": UNMAPPED_DOCUMENTED,
               "apps_without_score_reject": N_NO_SCORE, "spot_check": SPOT, "spot_check_ok": SPOT_OK,
               "swap_reconciles": SWAP_OK},
    "extrapolation_any": {f"{k[0]}/{k[1]}": v for k, v in EXT_ANY.items()},
    "outputs": sorted(set(RI_WRITTEN + [RI_SUMMARY])),
    "pass_fail": "PASS_WITH_DEVIATIONS" if (REG_OK and WRITE_OK and WF_REMOVED == 0 and N_NO_SCORE == 0 and SPOT_OK
                                           and SWAP_OK and UNMAPPED_DOCUMENTED) else "FAIL",
}
_tmp_js = os.path.join(RI_TMP, "ri_run_summary.json")
with open(_tmp_js, "w") as fh:
    json.dump(ri_summary, fh, indent=2, default=str)
shutil.copyfile(_tmp_js, ri_out(RI_SUMMARY))
print(f"Run summary: {RI_SUMMARY}")

assert REG_OK, "Registry, a locked model/binning file or the calibration file changed during the run"
assert WRITE_OK, f"Files changed outside {RI_DIR}: {CHANGED_OUTSIDE[:10]}"
assert WF_REMOVED == 0, f"Verification waterfall removed {WF_REMOVED} declined row(s)"
assert N_NO_SCORE == 0, f"{N_NO_SCORE} declined application(s) have no score_reject reason"
assert UNMAPPED_DOCUMENTED, "Unmapped reasons exist that are not listed in open_questions.md"
assert SPOT_OK, f"Approved scores do not reproduce the stored E13 scores: {SPOT}"
assert SWAP_OK, "Swap-in + swap-out + both-approve + both-decline do not reconcile to the totals"
print(f"\u2705 RI acceptance passed with documented deviations: {len(UNMAPPED)} unmapped reason(s) as PENDING, "
      f"{N_NOT_VERIFIABLE} filter rule(s) not verifiable. See {RI_OQ}")